In [ ]:
from dask.distributed import LocalCluster, Client
cluster = LocalCluster()
client = Client(cluster)

In [ ]:
foldername = '/home/edavenport/analysis/vel-assim-manuscript/assimilation_results/spectra/'

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.size'] = 13
import cmocean.cm as cmo
import xarray as xr
from open_tpose import tpose2012to2013
import numpy as np
import xarray as xr
from xmitgcm import open_mdsdataset

prefix = ['diag_state','diag_surf']
ds_tpose_noTAO = tpose2012to2013(prefix)

ds_tpose_noTAO['XC'] = ds_tpose_noTAO.XC.astype(float)
ds_tpose_noTAO['YC'] = ds_tpose_noTAO.YC.astype(float)
ds_tpose_noTAO['Z'] = ds_tpose_noTAO.Z.astype(float)
ds_tpose_noTAO['XG'] = ds_tpose_noTAO.XG.astype(float)
ds_tpose_noTAO['YC'] = ds_tpose_noTAO.YC.astype(float)

data_dir = '/data/SO3/edavenport/tpose6/sep2012/velocity_assim/run_iter22/'
grid_dir = '/data/SO6/TPOSE_diags/tpose6/grid_6/'

offset = 10
num_diags = 30+31+offset #sep, oct + 10 days
itPerFile = 72 # 1 day
intervals = range(itPerFile,itPerFile*num_diags,itPerFile)

ds = open_mdsdataset(data_dir=data_dir,grid_dir=grid_dir,iters=intervals,prefix=prefix,ref_date='2012-08-31 12:00:00',delta_t=1200)

num_diags = 30+31+offset# nov, dec (starting from nov 10)
itPerFile = 72 # 1 day
intervals = range(itPerFile*offset,itPerFile*num_diags,itPerFile)

data_dir = '/data/SO3/edavenport/tpose6/nov2012/run_iter20/'
ds_new = open_mdsdataset(data_dir=data_dir,grid_dir=grid_dir,iters=intervals,prefix=prefix,ref_date='2012-10-31 12:00:00',delta_t=1200)

ds_tpose_TAO = xr.concat([ds,ds_new],dim='time')

num_diags = 31+28+offset # jan, feb, (starting from jan 10) # add + offset to continue
itPerFile = 72 # 1 day
intervals = range(itPerFile*offset,itPerFile*num_diags,itPerFile)

data_dir = '/data/SO3/edavenport/tpose6/jan2013/run_iter14/'
ds_new = open_mdsdataset(data_dir=data_dir,grid_dir=grid_dir,iters=intervals,prefix=prefix,ref_date='2012-12-31 12:00:00',delta_t=1200)

ds_tpose_TAO = xr.concat([ds_tpose_TAO,ds_new],dim='time')

num_diags = 31+30+31+30+1 # mar, apr, may, june (starting from jan 10)
itPerFile = 72 # 1 day
intervals = range(itPerFile*offset,itPerFile*num_diags,itPerFile)

data_dir = '/data/SO3/edavenport/tpose6/mar2013/run_iter16/'
ds_new = open_mdsdataset(data_dir=data_dir,grid_dir=grid_dir,iters=intervals,prefix=prefix,ref_date='2013-03-01',delta_t=1200)

ds_tpose_TAO = xr.concat([ds_tpose_TAO,ds_new],dim='time')

ds_tpose_TAO['XC'] = ds_tpose_TAO.XC.astype(float)
ds_tpose_TAO['YC'] = ds_tpose_TAO.YC.astype(float)
ds_tpose_TAO['Z'] = ds_tpose_TAO.Z.astype(float)
ds_tpose_TAO['XG'] = ds_tpose_TAO.XG.astype(float)
ds_tpose_TAO['YC'] = ds_tpose_TAO.YC.astype(float)

In [ ]:
ds_tpose_noTAO = ds_tpose_noTAO.sel(time=slice('2012-09-01','2013-06-30'))

In [ ]:
def psd_for_plot_2d(data_array):
    # --- time spacing in seconds ---
    dt = (data_array.time[1] - data_array.time[0]).values / np.timedelta64(1, 's')
    fs = 1.0 / dt                     # sampling frequency (Hz)
    N = len(data_array)
    window = np.hanning(N)

    data_detrended = (data_array - data_array.mean(dim='time'))
    data_windowed = data_detrended * window[:,np.newaxis]
    fft_values = np.fft.rfft(data_windowed,axis=0)

    # --- PSD normalization ---
    U = (window**2).sum()       # window power
    psd_hz = (np.abs(fft_values) ** 2) / (U * fs)
    psd_hz[1:-1] *= 2 # variance preserving spectra
    psd_cpd = psd_hz / 86400
    freq_hz = np.fft.rfftfreq(N, d=dt)
    freq_dpc = 1/(freq_hz * 86400)        # cycles per day
    psd_dpc = psd_cpd / (freq_dpc**2)
    
    return fft_values, psd_hz, psd_dpc, freq_hz, freq_dpc

def psd_for_plot_1d(data_array):
    # --- time spacing in seconds ---
    dt = (data_array.time[1] - data_array.time[0]).values / np.timedelta64(1, 's')
    fs = 1.0 / dt                     # sampling frequency (Hz)
    N = len(data_array)
    window = np.hanning(N)

    data_detrended = (data_array - data_array.mean(dim='time'))
    data_windowed = data_detrended * window
    fft_values = np.fft.rfft(data_windowed,axis=0)

    # --- PSD normalization ---
    U = (window**2).sum()       # window power
    psd_hz = (np.abs(fft_values) ** 2) / (U * fs)
    psd_hz[1:-1] *= 2 # variance preserving spectra
    psd_cpd = psd_hz / 86400
    freq_hz = np.fft.rfftfreq(N, d=dt)
    freq_dpc = 1/(freq_hz * 86400)        # cycles per day
    psd_dpc = psd_cpd / (freq_dpc**2)
    
    return fft_values, psd_hz, psd_dpc, freq_hz, freq_dpc

In [ ]:
import xgcm 
grid = xgcm.Grid(ds_tpose_TAO, periodic=False)
UVEL = grid.interp(ds_tpose_TAO.UVEL, 'X', boundary='fill')
grid = xgcm.Grid(ds_tpose_noTAO, periodic=False)
UVEL_noTAO = grid.interp(ds_tpose_noTAO.UVEL, 'X', boundary='fill')

In [ ]:
grid = xgcm.Grid(ds_tpose_TAO, periodic=False)
VVEL = grid.interp(ds_tpose_TAO.VVEL, 'Y', boundary='fill')
grid = xgcm.Grid(ds_tpose_noTAO, periodic=False)
VVEL_noTAO = grid.interp(ds_tpose_noTAO.VVEL, 'Y', boundary='fill')

In [ ]:
from scipy import signal
def filter_psd_uvel_1d(X, Z):
    # filtering out high frequency changes
    fs = 1/86400 # sampling rate is 1 day (86400 seconds per day)
    highF = (1/10)*fs #  (1 cycle /35 days) * (1 day/86400 second)
    cutoff = np.array(highF)
    order = 4
    sos = signal.butter(order, cutoff, 'lowpass', fs=fs, output='sos')

    uvel_detrend = signal.detrend(UVEL.sel(YC=0.0, XC=X, Z=Z, method='nearest'), axis=0)
    uvel_filt = signal.sosfiltfilt(sos, uvel_detrend, axis=0)
    uvel_detrend_noTAO = signal.sosfiltfilt(sos, UVEL_noTAO.sel(YC=0.0, XC=X, Z=Z, method='nearest'), axis=0)
    uvel_filt_noTAO = signal.sosfiltfilt(sos, uvel_detrend_noTAO, axis=0)

    uvel_filt = xr.DataArray(
        uvel_filt,
        dims=('time'),
        coords={'time': UVEL.sel(YC=0.0, XC=X, Z=Z, method='nearest').time},
    )

    uvel_filt_noTAO = xr.DataArray(
        uvel_filt_noTAO,
        dims=('time'),
        coords={'time': UVEL.sel(YC=0.0, XC=X, Z=Z, method='nearest').time},
    )

    _, _, psd, _, freq_dpc = psd_for_plot_1d(uvel_filt)
    _, _, psd_noTAO, _, freq_dpc = psd_for_plot_1d(uvel_filt_noTAO)

    return psd, psd_noTAO, freq_dpc

def filter_psd_vvel_1d(X, Z):
    # filtering out high frequency changes
    fs = 1/86400 # sampling rate is 1 day (86400 seconds per day)
    highF = (1/10)*fs #  (1 cycle /35 days) * (1 day/86400 second)
    cutoff = np.array(highF)
    order = 4
    sos = signal.butter(order, cutoff, 'lowpass', fs=fs, output='sos')

    vvel_detrend = signal.detrend(VVEL.sel(YC=0.0, XC=X, Z=Z, method='nearest'), axis=0)
    vvel_filt = signal.sosfiltfilt(sos, vvel_detrend, axis=0)
    vvel_detrend_noTAO = signal.sosfiltfilt(sos, VVEL_noTAO.sel(YC=0.0, XC=X, Z=Z, method='nearest'), axis=0)
    vvel_filt_noTAO = signal.sosfiltfilt(sos, vvel_detrend_noTAO, axis=0)

    vvel_filt = xr.DataArray(
        vvel_filt,
        dims=('time'),
        coords={'time': VVEL.sel(YC=0.0, XC=X, Z=Z, method='nearest').time},
    )

    vvel_filt_noTAO = xr.DataArray(
        vvel_filt_noTAO,
        dims=('time'),
        coords={'time': VVEL.sel(YC=0.0, XC=X, Z=Z, method='nearest').time},
    )

    _, _, psd, _, freq_dpc = psd_for_plot_1d(vvel_filt)
    _, _, psd_noTAO, _, freq_dpc = psd_for_plot_1d(vvel_filt_noTAO)

    return psd, psd_noTAO, freq_dpc


In [ ]:
fig, ax = plt.subplots(figsize=(6,8))
psd, psd_noTAO, freq_dpc = filter_psd_uvel_1d(X=220.0, Z=-50.0)
ax.plot(freq_dpc, psd,label='50m, TAO')
ax.plot(freq_dpc, psd_noTAO,label='50m, No TAO')
psd, psd_noTAO, freq_dpc = filter_psd_uvel_1d(X=220.0, Z=-100.0)
ax.plot(freq_dpc, psd,label='100m, TAO',color='C0',linestyle='--')
ax.plot(freq_dpc, psd_noTAO,label='100m, No TAO',color='C1',linestyle='--')
psd, psd_noTAO, freq_dpc = filter_psd_uvel_1d(X=220.0, Z=-150.0)
ax.plot(freq_dpc, psd,label='150m, TAO',color='C0',linestyle=':')
ax.plot(freq_dpc, psd_noTAO,label='150m, No TAO',color='C1',linestyle=':')
ax.legend()
ax.set_xlabel('Frequency (cycles per day)')
ax.set_ylabel('Power (m$^2$/s$^2$/cpd)')
ax.loglog()
ax.set_xlim(2,150)
fig.savefig(foldername + 'U_power_spectrum_140W.png',dpi=300)

In [ ]:
fig, axes = plt.subplots(figsize=(10,10),nrows=3)

for ax, lon in zip(axes,[190, 220, 250]):
    psd, psd_noTAO, freq_dpc = filter_psd_uvel_1d(X=lon, Z=-50.0)
    ax.plot(freq_dpc, psd,label='50m, TAO')
    ax.plot(freq_dpc, psd_noTAO,label='50m, No TAO')
    psd, psd_noTAO, freq_dpc = filter_psd_uvel_1d(X=lon, Z=-100.0)
    ax.plot(freq_dpc, psd,label='100m, TAO',color='C0',linestyle='--')
    ax.plot(freq_dpc, psd_noTAO,label='100m, No TAO',color='C1',linestyle='--')
    psd, psd_noTAO, freq_dpc = filter_psd_uvel_1d(X=lon, Z=-150.0)
    ax.plot(freq_dpc, psd,label='150m, TAO',color='C0',linestyle=':')
    ax.plot(freq_dpc, psd_noTAO,label='150m, No TAO',color='C1',linestyle=':')
    ax.set_xlabel('')
    ax.set_ylabel('Power (m$^2$/s$^2$/cpd)')
    ax.set_title(f'0N, {np.abs(lon-360):.0f}W')
    ax.set_xlim(2,150)
    ax.loglog()

axes[0].legend(loc='lower right',ncol=2,fontsize=11)
axes[2].set_xlabel('Frequency (cycles per day)')
plt.tight_layout()
fig.savefig(foldername + 'U_power_spectrum.png',dpi=300)

In [ ]:
fig, axes = plt.subplots(figsize=(16,10),nrows=3,ncols=2)

for ax, lon in zip(axes.flatten()[0::2],[190, 220, 250]):
    psd, psd_noTAO, freq_dpc = filter_psd_uvel_1d(X=lon, Z=-50.0)
    ax.plot(freq_dpc, psd,label='50m, TAO')
    ax.plot(freq_dpc, psd_noTAO,label='50m, No TAO')
    psd, psd_noTAO, freq_dpc = filter_psd_uvel_1d(X=lon, Z=-100.0)
    ax.plot(freq_dpc, psd,label='100m, TAO',color='C0',linestyle='--')
    ax.plot(freq_dpc, psd_noTAO,label='100m, No TAO',color='C1',linestyle='--')
    psd, psd_noTAO, freq_dpc = filter_psd_uvel_1d(X=lon, Z=-150.0)
    ax.plot(freq_dpc, psd,label='150m, TAO',color='C0',linestyle=':')
    ax.plot(freq_dpc, psd_noTAO,label='150m, No TAO',color='C1',linestyle=':')
    ax.set_xlabel('')
    ax.set_ylabel('Power (m$^2$/s$^2$/cpd)')
    ax.set_title(f'0N, {np.abs(lon-360):.0f}W U')
    ax.loglog()
    ax.set_xlim(10,150)
    ax.set_ylim(1e-7,1e-2)

for ax, lon in zip(axes.flatten()[1::2],[190, 220, 250]):
    psd, psd_noTAO, freq_dpc = filter_psd_vvel_1d(X=lon, Z=-50.0)
    ax.plot(freq_dpc, psd,label='50m, TAO')
    ax.plot(freq_dpc, psd_noTAO,label='50m, No TAO')
    psd, psd_noTAO, freq_dpc = filter_psd_vvel_1d(X=lon, Z=-100.0)
    ax.plot(freq_dpc, psd,label='100m, TAO',color='C0',linestyle='--')
    ax.plot(freq_dpc, psd_noTAO,label='100m, No TAO',color='C1',linestyle='--')
    psd, psd_noTAO, freq_dpc = filter_psd_vvel_1d(X=lon, Z=-150.0)
    ax.plot(freq_dpc, psd,label='150m, TAO',color='C0',linestyle=':')
    ax.plot(freq_dpc, psd_noTAO,label='150m, No TAO',color='C1',linestyle=':')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(f'0N, {np.abs(lon-360):.0f}W V')
    ax.loglog()
    ax.set_xlim(10,150)
    ax.set_ylim(1e-7,1e-2)

axes[0,0].legend(loc='lower right',ncol=3,fontsize=11)
axes[2,0].set_xlabel('Frequency (cycles per day)')
axes[2,1].set_xlabel('Frequency (cycles per day)')
plt.tight_layout()
fig.savefig(foldername + 'UV_power_spectrum_10day.png',dpi=300)

In [ ]:
import numba
from numba import njit

@njit
def test_func(x):
    return x + 1

print(test_func(1))

In [ ]:
%conda install numba
grid = xgcm.Grid(ds_tpose_TAO, periodic=False)

# create grid
target_values = np.arange(14, 23.5, 0.1)

theta_interp = grid.interp(ds_tpose_TAO.THETA, 'X', boundary='fill')
UVEL_theta = grid.transform(ds_tpose_TAO.UVEL, axis='Z', target=target_values, target_data=theta_interp, method='linear').transpose('time','THETA','YC','XG')
theta_interp = grid.interp(ds_tpose_TAO.THETA, 'Y', boundary='fill')
VVEL_theta = grid.transform(ds_tpose_TAO.VVEL, axis='Z', target=target_values, target_data=theta_interp, method='linear').transpose('time','THETA','YG','XC')

thetaMin = 20.0
depthui = np.argmin(np.abs(target_values - thetaMin))
u_theta = UVEL_theta[:,depthui]

# fig, ax = plt.subplots(figsize=(10,8),nrows=2)
# u_var_transect.plot(ax=ax[0], cmap=cmo.haline, vmin=0.0, vmax=0.075,cbar_kwargs={'label': '$m^2/s^2$'})
# ax[0].set_title('var(UVEL), sigma_0=24.5')
# v_var_transect.plot(ax=ax[1], cmap=cmo.haline, vmin=0.0, vmax=0.075,cbar_kwargs={'label': '$m^2/s^2$'})
# ax[1].set_title('var(VVEL), sigma_0=24.5')
# plt.tight_layout()

In [ ]:
from scipy import signal
def filter_psd_uvel_isotherm(X, T):
    # filtering out high frequency changes
    fs = 1/86400 # sampling rate is 1 day (86400 seconds per day)
    highF = (1/10)*fs #  (1 cycle /35 days) * (1 day/86400 second)
    cutoff = np.array(highF)
    order = 4
    sos = signal.butter(order, cutoff, 'lowpass', fs=fs, output='sos')

    uvel_detrend = signal.detrend(UVEL.sel(YC=0.0, XC=X, THETA=T, method='nearest'), axis=0)
    uvel_filt = signal.sosfiltfilt(sos, uvel_detrend, axis=0)
    uvel_detrend_noTAO = signal.sosfiltfilt(sos, UVEL_noTAO.sel(YC=0.0, XC=X, THETA=T, method='nearest'), axis=0)
    uvel_filt_noTAO = signal.sosfiltfilt(sos, uvel_detrend_noTAO, axis=0)

    uvel_filt = xr.DataArray(
        uvel_filt,
        dims=('time'),
        coords={'time': UVEL.sel(YC=0.0, XC=X, THETA=T, method='nearest').time},
    )

    uvel_filt_noTAO = xr.DataArray(
        uvel_filt_noTAO,
        dims=('time'),
        coords={'time': UVEL.sel(YC=0.0, XC=X, THETA=T, method='nearest').time},
    )

    _, _, psd, _, freq_dpc = psd_for_plot_1d(uvel_filt)
    _, _, psd_noTAO, _, freq_dpc = psd_for_plot_1d(uvel_filt_noTAO)

    return psd, psd_noTAO, freq_dpc

def filter_psd_vvel_isotherm(X, T):
    # filtering out high frequency changes
    fs = 1/86400 # sampling rate is 1 day (86400 seconds per day)
    highF = (1/10)*fs #  (1 cycle /35 days) * (1 day/86400 second)
    cutoff = np.array(highF)
    order = 4
    sos = signal.butter(order, cutoff, 'lowpass', fs=fs, output='sos')

    vvel_detrend = signal.detrend(VVEL.sel(YC=0.0, XC=X, THETA=T, method='nearest'), axis=0)
    vvel_filt = signal.sosfiltfilt(sos, vvel_detrend, axis=0)
    vvel_detrend_noTAO = signal.sosfiltfilt(sos, VVEL_noTAO.sel(YC=0.0, XC=X, THETA=T, method='nearest'), axis=0)
    vvel_filt_noTAO = signal.sosfiltfilt(sos, vvel_detrend_noTAO, axis=0)

    vvel_filt = xr.DataArray(
        vvel_filt,
        dims=('time'),
        coords={'time': VVEL.sel(YC=0.0, XC=X, THETA=T, method='nearest').time},
    )

    vvel_filt_noTAO = xr.DataArray(
        vvel_filt_noTAO,
        dims=('time'),
        coords={'time': VVEL.sel(YC=0.0, XC=X, THETA=T, method='nearest').time},
    )

    _, _, psd, _, freq_dpc = psd_for_plot_1d(vvel_filt)
    _, _, psd_noTAO, _, freq_dpc = psd_for_plot_1d(vvel_filt_noTAO)

    return psd, psd_noTAO, freq_dpc


In [ ]:
client.shutdown()
cluster.close()
client.close()